# MOT17 Pipeline Configurations

https://github.com/Sreebhargavibalijaa/CSE8001---Object-Detection.git

This notebook runs the detection -> tracking -> evaluation pipeline for the different
detector + tracker configurations and collects the metrics into a single comparison table.

```text
MOT17 images -> detect.py -> detections CSV -> track.py -> MOT track file -> evaluate.py -> metrics
```

Before running: install the dependencies and place the MOT17 dataset as described in
the README, then run this notebook from the repo root

## 1. Configurations

This configuration is one detector paired with one tracker. The lists below define which
ones to run. With 3 detectors and 2 trackers there are 6 possible configurations. On CPU, YOLO26n is fast but Faster R-CNN and DETR are slow, so this defaults to a short test (MAX_FRAMES = 30)

In [ ]:
import sys
import subprocess
from pathlib import Path

# --- edit these to choose what to run ---
#DETECTORS = ["yolo26n", "fasterrcnn", "detr"]   # any of: yolo26n, fasterrcnn, detr
#TRACKERS  = ["bytetrack", "ocsort"]              # any of: bytetrack, ocsort  
DETECTORS = ["yolo26n"]
TRACKERS  = ["bytetrack"]  
SEQUENCE  = "MOT17-02-DPM"                        # any folder under MOT17/train
MAX_FRAMES = 30                                   # set to None for the whole video
# ----------------------------------------

PIPELINE_DIR = Path.cwd() / "mot_pipeline (Object_Detection + Tracking)"
OUTPUT_ROOT = PIPELINE_DIR / "outputs"
PY = sys.executable                # the Python running this notebook

assert PIPELINE_DIR.is_dir(), f"Run this notebook from the repo root; not found: {PIPELINE_DIR}"
configs = [(d, t) for d in DETECTORS for t in TRACKERS]
print("Sequence:", SEQUENCE, "| max frames:", MAX_FRAMES)
print("Configurations to run:")
for d, t in configs:
    print(f"  - {d} + {t}")

Sequence: MOT17-02-DPM | max frames: 30
Configurations to run:
  - yolo26n + bytetrack


In [5]:
def run(script, extra_args):
    """Call one pipeline script and stream its output, exactly like the command line."""
    cmd = [PY, script, "--sequence", SEQUENCE]
    if MAX_FRAMES is not None:
        cmd += ["--max-frames", str(MAX_FRAMES)]
    cmd += extra_args
    print(">", " ".join(cmd))
    subprocess.run(cmd, cwd=PIPELINE_DIR, check=True)

## 2. Detect

Each detector is run once over the sequence; both trackers later reuse the same boxes.

In [6]:
for detector in DETECTORS:
    run("detect.py", ["--detector", detector])

> c:\Python314\python.exe detect.py --sequence MOT17-02-DPM --max-frames 30 --detector yolo26n


## 3. Track

Run every detector + tracker configuration on the saved detections.

In [7]:
for detector, tracker in configs:
    detections = OUTPUT_ROOT / "detections" / SEQUENCE / f"{detector}.csv"
    run("track.py", ["--tracker", tracker, "--detections", str(detections)])

> c:\Python314\python.exe track.py --sequence MOT17-02-DPM --max-frames 30 --tracker bytetrack --detections c:\Users\Preya\Desktop\Mizzou\Sem3\mlops\project\CSE8001---Object-Detection\mot_pipeline (Object_Detection + Tracking)\outputs\detections\MOT17-02-DPM\yolo26n.csv


## 4. Evaluate

Score each configuration's track file with the official TrackEval metrics.

In [8]:
for detector, tracker in configs:
    tracks = OUTPUT_ROOT / "tracks" / SEQUENCE / f"{detector}_{tracker}.txt"
    run("evaluate.py", ["--tracks", str(tracks)])

> c:\Python314\python.exe evaluate.py --sequence MOT17-02-DPM --max-frames 30 --tracks c:\Users\Preya\Desktop\Mizzou\Sem3\mlops\project\CSE8001---Object-Detection\mot_pipeline (Object_Detection + Tracking)\outputs\tracks\MOT17-02-DPM\yolo26n_bytetrack.txt


## 5. Compare the configurations

In [9]:
import pandas as pd

rows = []
for detector, tracker in configs:
    csv_path = OUTPUT_ROOT / "metrics" / SEQUENCE / f"{detector}_{tracker}.csv"
    row = pd.read_csv(csv_path).iloc[0].to_dict()
    row = {"detector": detector, "tracker": tracker, **row}
    rows.append(row)

summary = pd.DataFrame(rows)
# show the headline metrics first; the full set is still in the DataFrame
headline = ["detector", "tracker", "HOTA", "MOTA", "IDF1", "DetA", "AssA"]
summary[headline].round(2)

,detector,tracker,HOTA,MOTA,IDF1,DetA,AssA
0,yolo26n,bytetrack,40.47,24.12,40.69,20.88,79.58


In [10]:
# full metrics for every configuration
summary.round(3)

,detector,tracker,sequence,tracks,frames,HOTA,DetA,AssA,LocA,MOTA,...,IDR,FP,FN,IDSW,Frag,Precision,Recall,assertions_checked,assertions_failed,assertion_pass_rate
0,yolo26n,bytetrack,MOT17-02-DPM,yolo26n_bytetrack,30,40.473,20.885,79.578,82.868,24.118,...,26.029,13,503,0,3,93.158,26.029,208,0,100.0
